# P08B — Discovery dan inferensi kausal

Jalankan P08A dahulu. ATE = E[Y(1)-Y(0)] untuk IRI enam bulan. Nilai negatif berarti kondisi lebih baik.

In [ ]:
from pathlib import Path
import pandas as pd
f=Path('Bab8_Paket/Data/maintenance_synthetic.csv'); print(pd.read_csv(f).describe().round(2))

## Estimasi intervensi: pemeliharaan ruas jalan

Confounder terukur: IRI awal, proporsi truk, curah hujan, risiko drainase. DGP memasukkan efek treatment −0,80 IRI. Perbedaan mentah bukan efek kausal.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.linear_model import LinearRegression
f=Path('Bab8_Paket/Data/maintenance_synthetic.csv')
if not f.exists(): f=Path('maintenance_synthetic.csv')
if not f.exists():
 try:
  from google.colab import files
  u=files.upload(); f=Path(next(iter(u)))
 except Exception as e: raise FileNotFoundError('Jalankan P08A atau unggah CSV') from e
df=pd.read_csv(f); C=['iri_baseline','truck_share','annual_rainfall_mm','drainage_risk']; assert df.notna().all().all()
crude=df[df.maintenance==1].iri_6_months.mean()-df[df.maintenance==0].iri_6_months.mean()
X=df[C+['maintenance']]; m=LinearRegression().fit(X,df.iri_6_months); x1=X.copy(); x0=X.copy(); x1.maintenance=1; x0.maintenance=0
ate=np.mean(m.predict(x1)-m.predict(x0)); print('N=',len(df),'raw difference=',round(crude,3),'standardized estimate=',round(ate,3),'DGP truth=-0.80')
rng=np.random.default_rng(20261116); est=[]
for _ in range(500):
 s=df.iloc[rng.integers(0,len(df),len(df))]; xx=s[C+['maintenance']]; mm=LinearRegression().fit(xx,s.iri_6_months); a=xx.copy(); b=xx.copy(); a.maintenance=1; b.maintenance=0; est.append(np.mean(mm.predict(a)-mm.predict(b)))
print('Bootstrap 95% percentile CI:',np.quantile(est,[.025,.975]).round(3))
print('Bootstrap quantifies sampling variation only; it does not remove unmeasured confounding.')

## Discovery sebagai ilustrasi terbatas

Skeleton PC berikut menggunakan uji Fisher-Z. Kode tidak menyelesaikan orientasi dan tidak dimaksudkan sebagai implementasi penelitian. Asumsi discovery meliputi kausal sufficiency, faithfulness, tidak ada siklus, sampel memadai, dan uji independensi yang sesuai. Arah collider diketahui dari DGP dan mekanika.

In [ ]:
# Demonstrasi skeleton PC sangat terbatas pada tiga variabel kontinu.
# Uji Fisher-Z dan pencarian subset conditioning; tidak mengorientasikan edge, bukan implementasi produksi.
import numpy as np,pandas as pd
from itertools import combinations
from scipy.stats import norm
r=np.random.default_rng(8); n=1500; Z=r.lognormal(0,.35,n); fy=r.lognormal(0,.18,n); M=Z*fy+r.normal(0,.015,n); beam=pd.DataFrame({'Z':Z,'fy':fy,'M':M})
def pcorr(a,b,cond):
 x=beam[a].values; y=beam[b].values
 if cond:
  c=np.column_stack([np.ones(n),beam[list(cond)].values]); x=x-c@np.linalg.lstsq(c,x,rcond=None)[0]; y=y-c@np.linalg.lstsq(c,y,rcond=None)[0]
 rho=np.corrcoef(x,y)[0,1]; z=abs(np.arctanh(np.clip(rho,-.99999,.99999)))*np.sqrt(n-len(cond)-3); return rho,2*(1-norm.cdf(z))
edges={frozenset(e) for e in combinations(beam.columns,2)}
for x,y in combinations(beam.columns,2):
 for cond in [()]+[(q,) for q in beam.columns if q not in (x,y)]:
  if pcorr(x,y,cond)[1]>.05: edges.discard(frozenset((x,y))); break
print('Skeleton:',[sorted(e) for e in edges]); print('No arrow orientation; collider Z -> M <- fy comes from known DGP/domain, not this skeleton.')

## Refleksi
- Apakah treatment didefinisikan sebagai jenis/intensitas/waktu pekerjaan tertentu?
- Confounder apa yang mungkin tidak diukur (mis. kualitas kontraktor)?
- Apakah overlap cukup dan apakah ada spillover antarruas?
- Bagaimana nonlinearitas atau confounder hipotetis memengaruhi kesimpulan?
- Bootstrap mengukur variasi sampling, bukan bias identifikasi. DoWhy opsional untuk alur model–identify–estimate–refute; API dan versi perlu dicatat.